In [ ]:
from huggingface_hub import snapshot_download

# Change MODEL_ID to any public HuggingFace model repo
MODEL_ID   = 'meta-llama/Llama-3.1-8B-Instruct'
LOCAL_DIR  = f'./models/{MODEL_ID.split("/")[-1]}'

# Downloads all shards to LOCAL_DIR; skips files already present
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=LOCAL_DIR,
    ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],  # skip non-PyTorch weights
)

print(f'Model saved to: {LOCAL_DIR}')


# Transformers: From Scratch → Large Models
**Stack:** PyTorch 2.x · HuggingFace Transformers · RTX 5080

### Sections
1. GPU setup & VRAM check
2. Transformer built from scratch (PyTorch only)
3. HuggingFace pipeline basics
4. Large model loading — Llama / Mistral / Qwen (bfloat16, 4-bit)
5. Streaming generation & sampling strategies
6. Embedding models & semantic search

## 1 · GPU Setup

In [ ]:
import torch

assert torch.cuda.is_available(), 'CUDA not available — check your driver'

device = torch.device('cuda')
props  = torch.cuda.get_device_properties(device)

print(f'GPU  : {props.name}')
print(f'VRAM : {props.total_memory / 1e9:.1f} GB')
print(f'CUDA : {torch.version.cuda}')
print(f'torch: {torch.__version__}')
print(f'bf16 : {torch.cuda.is_bf16_supported()}')

## 2 · Transformer from Scratch (PyTorch)

### 2.1 Scaled Dot-Product Attention
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(
    q: torch.Tensor,   # (B, heads, T, d_k)
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
    dropout: float = 0.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    d_k = q.size(-1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)   # (B, H, T, T)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    if dropout > 0.0:
        weights = F.dropout(weights, p=dropout)
    return weights @ v, weights   # output, attention map


# Quick sanity check
B, H, T, d_k = 2, 4, 8, 16
q = k = v = torch.randn(B, H, T, d_k)
out, attn = scaled_dot_product_attention(q, k, v)
print('output shape  :', out.shape)    # (2, 4, 8, 16)
print('attn weights  :', attn.shape)   # (2, 4, 8, 8)

### 2.2 Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k    = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = dropout

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        B, T, _ = q.shape
        q = self._split_heads(self.W_q(q))
        k = self._split_heads(self.W_k(k))
        v = self._split_heads(self.W_v(v))

        x, _ = scaled_dot_product_attention(q, k, v, mask, self.dropout)
        x = x.transpose(1, 2).contiguous().view(B, T, -1)   # merge heads
        return self.W_o(x)


mha = MultiHeadAttention(d_model=256, n_heads=8)
x   = torch.randn(2, 10, 256)
print('MHA output:', mha(x, x, x).shape)   # (2, 10, 256)

### 2.3 Positional Encoding

In [ ]:
import matplotlib.pyplot as plt

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(x + self.pe[:, :x.size(1)])


pe   = SinusoidalPositionalEncoding(128, 100)
vals = pe.pe[0].detach().numpy()
plt.figure(figsize=(12, 4))
plt.imshow(vals.T, aspect='auto', cmap='RdBu')
plt.colorbar(); plt.xlabel('position'); plt.ylabel('dimension')
plt.title('Sinusoidal Positional Encoding'); plt.tight_layout(); plt.show()

### 2.4 Transformer Block & Full Decoder-Only Model

In [ ]:
class TransformerBlock(nn.Module):
    """Pre-norm decoder block (GPT-style)."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1   = nn.LayerNorm(d_model)
        self.ln2   = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        x = x + self.drop(self.attn(self.ln1(x), self.ln1(x), self.ln1(x), mask))
        x = x + self.drop(self.ff(self.ln2(x)))
        return x


class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model:    int  = 256,
        n_heads:    int  = 8,
        n_layers:   int  = 4,
        d_ff:       int  = 1024,
        max_len:    int  = 512,
        dropout:    float = 0.1,
    ):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pe    = SinusoidalPositionalEncoding(d_model, max_len, dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.ln_f  = nn.LayerNorm(d_model)
        self.head  = nn.Linear(d_model, vocab_size, bias=False)
        self.embed.weight = self.head.weight   # weight tying

    def _causal_mask(self, T: int, device: torch.device) -> torch.Tensor:
        return torch.tril(torch.ones(T, T, device=device)).unsqueeze(0).unsqueeze(0)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        T    = idx.size(1)
        mask = self._causal_mask(T, idx.device)
        x    = self.pe(self.embed(idx))
        for block in self.blocks:
            x = block(x, mask)
        return self.head(self.ln_f(x))   # (B, T, vocab_size)


model = MiniGPT(vocab_size=50257)
params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {params:.1f}M')

dummy  = torch.randint(0, 50257, (2, 32))
logits = model(dummy)
print(f'Logits shape: {logits.shape}')   # (2, 32, 50257)

### 2.5 Autoregressive Generation (greedy & top-p)


In [ ]:
@torch.no_grad()
def generate(
    model:       nn.Module,
    idx:         torch.Tensor,   # (1, T) seed tokens
    max_new:     int   = 50,
    temperature: float = 1.0,
    top_p:       float = 0.9,    # nucleus sampling
) -> torch.Tensor:
    model.eval()
    for _ in range(max_new):
        logits = model(idx)[:, -1, :] / temperature   # (1, V)

        # top-p (nucleus) sampling
        probs  = F.softmax(logits, dim=-1)
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cum    = sorted_probs.cumsum(dim=-1)
        remove = cum - sorted_probs > top_p
        sorted_probs[remove] = 0
        sorted_probs /= sorted_probs.sum()
        next_token = sorted_idx.gather(-1, torch.multinomial(sorted_probs, 1))

        idx = torch.cat([idx, next_token], dim=1)
    return idx


seed   = torch.randint(0, 50257, (1, 5))
result = generate(model, seed, max_new=20)
print('generated token ids:', result[0].tolist())

## 3 · HuggingFace Pipelines (Quick Start)

In [ ]:
from transformers import pipeline

# Fast text-gen with GPT-2 to verify the stack
gen = pipeline('text-generation', model='gpt2', device=0)
out = gen('The transformer architecture revolutionized NLP because', max_new_tokens=60, do_sample=True, top_p=0.9)
print(out[0]['generated_text'])

## 4 · Large Models on the RTX 5080

The RTX 5080 has **16 GB VRAM**. Rough VRAM budgets:

| Model | bfloat16 | 4-bit (NF4) |
|---|---|---|
| Llama-3.1-8B | ~16 GB | ~5 GB |
| Mistral-7B-v0.3 | ~14 GB | ~5 GB |
| Qwen2.5-14B | OOM | ~9 GB |
| Qwen2.5-32B | OOM | ~20 GB (OOM) |

We'll load with `bfloat16` by default, or swap to 4-bit via BitsAndBytes.

### 4.1 Llama 3.1 8B — bfloat16 (fits on 5080)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
# Requires HF token: huggingface-cli login
# Or set: HUGGINGFACE_TOKEN env var

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model_llama = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',          # fills GPU first, spills to CPU if needed
    attn_implementation='flash_attention_2',  # faster on Blackwell/Ampere+
)
model_llama.eval()
print(model_llama.hf_device_map)

In [ ]:
# Chat with Llama 3.1
messages = [
    {'role': 'system', 'content': 'You are a concise AI assistant.'},
    {'role': 'user',   'content': 'Explain attention mechanisms in 3 bullet points.'},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to(model_llama.device)

with torch.no_grad():
    out = model_llama.generate(
        inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
    )

response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print(response)

### 4.2 4-bit Quantization with BitsAndBytes (for 14B+ models)

In [ ]:
# pip install bitsandbytes  (if not installed)
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # NormalFloat4 — best quality
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # nested quantization saves ~0.4 GB
)

MODEL_14B = 'Qwen/Qwen2.5-14B-Instruct'

tokenizer_qwen = AutoTokenizer.from_pretrained(MODEL_14B)
model_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_14B,
    quantization_config=bnb_config,
    device_map='auto',
)
model_qwen.eval()

vram_used = torch.cuda.memory_allocated() / 1e9
print(f'VRAM used: {vram_used:.1f} GB')

### 4.3 Mistral 7B v0.3

In [ ]:
MODEL_MISTRAL = 'mistralai/Mistral-7B-Instruct-v0.3'

tokenizer_mistral = AutoTokenizer.from_pretrained(MODEL_MISTRAL)
model_mistral = AutoModelForCausalLM.from_pretrained(
    MODEL_MISTRAL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model_mistral.eval()
print('Mistral loaded')

## 5 · Streaming Generation

In [ ]:
from transformers import TextStreamer

def chat_stream(model, tokenizer, user_msg: str, system: str = '', max_new_tokens: int = 512):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': user_msg})

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            streamer=streamer,
        )

# Example — swap model_llama for model_mistral or model_qwen as needed
# chat_stream(model_llama, tokenizer, 'What is the key insight behind the transformer architecture?')

## 6 · Sampling Strategies Compared

In [ ]:
from transformers import GenerationConfig

PROMPT = 'Once upon a time in a land of silicon and light'

strategies = {
    'Greedy':       GenerationConfig(do_sample=False),
    'Beam (n=4)':   GenerationConfig(do_sample=False, num_beams=4),
    'Top-k (k=50)': GenerationConfig(do_sample=True, top_k=50, temperature=1.0),
    'Top-p (0.92)': GenerationConfig(do_sample=True, top_p=0.92, temperature=1.0),
    'Temp 0.3':     GenerationConfig(do_sample=True, temperature=0.3),
    'Temp 1.5':     GenerationConfig(do_sample=True, temperature=1.5),
}

# Use GPT-2 for fast comparison across strategies
from transformers import GPT2LMHeadModel, GPT2Tokenizer
tok2  = GPT2Tokenizer.from_pretrained('gpt2')
gpt2  = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
gpt2.eval()

inp = tok2(PROMPT, return_tensors='pt').input_ids.to(device)

for name, cfg in strategies.items():
    with torch.no_grad():
        out = gpt2.generate(inp, generation_config=cfg, max_new_tokens=40, pad_token_id=tok2.eos_token_id)
    text = tok2.decode(out[0], skip_special_tokens=True)
    print(f'\n[{name}]\n{text}')

## 7 · Embedding Models & Semantic Search

In [ ]:
from transformers import AutoModel

EMBED_MODEL = 'BAAI/bge-large-en-v1.5'   # 1.3B params, 1024-dim

embed_tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(
    EMBED_MODEL, torch_dtype=torch.bfloat16
).to(device).eval()


def embed(texts: list[str]) -> torch.Tensor:
    enc = embed_tok(texts, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
    with torch.no_grad():
        out = embed_model(**enc)
    # CLS-token pooling
    emb = out.last_hidden_state[:, 0]
    return F.normalize(emb, dim=-1)   # unit vectors for cosine sim


docs = [
    'Transformers use self-attention to process sequences.',
    'PyTorch is a deep learning framework backed by Meta.',
    'The Eiffel Tower is located in Paris, France.',
    'BERT is a bidirectional encoder pre-trained on masked language modeling.',
    'Gradient descent optimizes neural network weights iteratively.',
]

query = 'How does the attention mechanism work in neural networks?'

doc_embs   = embed(docs)
query_emb  = embed([query])
sims       = (query_emb @ doc_embs.T).squeeze(0)

ranked = sims.argsort(descending=True)
print(f'Query: {query}\n')
for i, idx in enumerate(ranked):
    print(f'{i+1}. [{sims[idx]:.3f}] {docs[idx]}')

## 8 · Visualising Attention Weights

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import GPT2Model, GPT2Tokenizer

tok_vis = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_vis = GPT2Model.from_pretrained('gpt2', output_attentions=True).to(device).eval()

sentence = 'The cat sat on the mat because it was tired'
ids      = tok_vis(sentence, return_tensors='pt').input_ids.to(device)
tokens   = tok_vis.convert_ids_to_tokens(ids[0])

with torch.no_grad():
    outputs = gpt2_vis(ids)

# Last layer, all 12 heads averaged
attn_last = outputs.attentions[-1][0].cpu().float()   # (12, T, T)
attn_mean = attn_last.mean(0).numpy()                 # (T, T)

plt.figure(figsize=(9, 7))
sns.heatmap(
    attn_mean, xticklabels=tokens, yticklabels=tokens,
    cmap='Blues', linewidths=0.3, annot=False
)
plt.title('GPT-2 — Last Layer Attention (avg over 12 heads)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## Tips for the RTX 5080

```python
# Enable TF32 for matmuls — free speed on Ampere+
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

# Compile model for ~30% throughput gain (PyTorch 2+)
model = torch.compile(model, mode='reduce-overhead')

# Flash Attention 2 — pass to from_pretrained:
# attn_implementation='flash_attention_2'

# Monitor VRAM
print(torch.cuda.memory_summary(abbreviated=True))
```